In [0]:
%sql
USE CATALOG `retail-sales`;
USE SCHEMA `03_gold`;

In [0]:
%sql
-- FactSales
CREATE TABLE IF NOT EXISTS `retail-sales`.`03_gold`.FactSales (
  SalesSK       BIGINT GENERATED ALWAYS AS IDENTITY,
  TransactionID INT,
  CustomerSK    BIGINT,
  ProductSK     BIGINT,
  StoreSK       BIGINT,
  Quantity      INT,
  Amount        DECIMAL(12,2),
  TxnDate       DATE
) USING DELTA LOCATION 's3://retail-sales-data-wh/processed/FactSales/';

In [0]:
%sql
INSERT INTO `retail-sales`.`03_gold`.FactSales (
  TransactionID, CustomerSK, ProductSK, StoreSK,
  Quantity, Amount, TxnDate
)
WITH clean_sales AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY TransactionID ORDER BY TxnDate) AS rn
  FROM `retail-sales`.01_bronze.sales_raw
  WHERE Quantity       > 0
    AND CustomerID    <= 500
    AND TransactionID <= 10000
)
SELECT
  s.TransactionID,
  c.CustomerSK,
  p.ProductSK,
  st.StoreSK,
  s.Quantity,
  CAST(s.Quantity * p.UnitPrice AS DECIMAL(12,2)) AS Amount,
  CAST(s.TxnDate AS DATE)                         AS TxnDate
FROM clean_sales                              AS s
JOIN `retail-sales`.02_silver.DimCustomer    AS c
  ON s.CustomerID = c.CustomerID AND c.IsActive = 1
JOIN `retail-sales`.02_silver.DimProduct     AS p
  ON s.ProductID  = p.ProductID
JOIN `retail-sales`.02_silver.DimStore       AS st
  ON s.StoreID    = st.StoreID
WHERE s.rn = 1;

In [0]:
%sql
SELECT 'DimCustomer' AS table_name, COUNT(*) AS total_rows,
  SUM(CASE WHEN IsActive = 1 THEN 1 ELSE 0 END) AS active_records,
  SUM(CASE WHEN IsActive = 0 THEN 1 ELSE 0 END) AS expired_records
FROM `retail-sales`.02_silver.DimCustomer
UNION ALL
SELECT 'DimProduct', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimProduct
UNION ALL
SELECT 'DimStore', COUNT(*), NULL, NULL
FROM `retail-sales`.02_silver.DimStore
UNION ALL
SELECT 'FactSales', COUNT(*), NULL, NULL
FROM `retail-sales`.`03_gold`.FactSales;